# ICOICT 2026 – Travlr Data Challenge
## Hotel Recommendation System Based on Popularity, Rating, and Price Analysis

**Name:** Joyce Stephanie Naibaho
**Institution:** Institut Teknologi Del (IT Del), Indonesia
**Date:** May 17, 2026

---
### Description
This notebook contains:
1. Load and exploration of 3 Travlr datasets (EDA)
2. Data preprocessing and cleaning
3. Composite Scoring Model for hotel recommendation (Popularity + Rating + ReviewCount + Price)
4. Content-Based Filtering (TF-IDF) as a comparison method
5. Evaluation using standard metrics: Precision@K, Recall@K, NDCG@K
6. Visualization results (8 separate EDA figures + 1 method comparison chart)
7. Insights and conclusion

### Evaluation Results @K=10
| Method | Precision@10 | Recall@10 | NDCG@10 |
|---|---|---|---|
| Composite Scoring (Ours) | **0.8000** | **0.0104** | **0.8572** |
| Content-Based Filtering (TF-IDF) | 0.2000 | 0.0026 | 0.2048 |

Composite Scoring is **4× superior** across all metrics.

## 0. Install Required Libraries

In [40]:
# Run this cell only once if the libraries are not installed yet
# !pip install pandas numpy matplotlib seaborn openpyxl

## 1. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)

FILE_PATH = 'Dataset.xlsx'

df_meta = pd.read_excel(FILE_PATH, sheet_name='accommodation metadata dataset')
df_act  = pd.read_excel(FILE_PATH, sheet_name='Activity Dataset')
df_tx   = pd.read_excel(FILE_PATH, sheet_name='accomodation transaction datase')

print(f'Metadata : {df_meta.shape[0]} rows, {df_meta.shape[1]} columns')
print(f'Activity : {df_act.shape[0]} rows, {df_act.shape[1]} columns')
print(f'Transaction: {df_tx.shape[0]} rows, {df_tx.shape[1]} columns')

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Display first 5 rows of each dataset ──
print('METADATA AKOMODASI')
display(df_meta[['id','name','country','city','category','star_rating',
                  'guest_rating_score','popularity_score','price_in_aud']].head())

print('\nTRANSAKSI AKOMODASI')
display(df_tx.head())

print('\nAKTIVITAS WISATA')
display(df_act[['name','categories_as_string','destination_country',
                 'price_in_aud','rating','duration_hours']].head())

In [ ]:
# ── Check missing values ──
print('MISSING VALUES – Metadata Akomodasi')
missing_meta = df_meta.isnull().sum()
display(missing_meta[missing_meta > 0].sort_values(ascending=False))

print('\nMISSING VALUES – Transaksi')
print(df_tx.isnull().sum())

print('\nMISSING VALUES – Aktivitas')
missing_act = df_act.isnull().sum()
display(missing_act[missing_act > 0].sort_values(ascending=False))

In [ ]:
# ── Descriptive statistics of metadata ──
print('DESCRIPTIVE STATISTICS – Accommodation Metadata')
display(df_meta[['star_rating','guest_rating_score','popularity_score','price_in_aud']].describe())

## 3. EDA Visualization

In [ ]:
# Fig 1 – Top 10 Countries – Number of Accommodations
fig, ax = plt.subplots(figsize=(9, 5))
top_countries = df_meta['country'].value_counts().head(10)
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#378ADD', edgecolor='none', height=0.6)
ax.set_title('Top 10 Countries – Number of Accommodations', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Hotels', fontsize=11)
ax.spines[['top','right']].set_visible(False)
for i, v in enumerate(top_countries.values[::-1]):
    ax.text(v+10, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_country.png')

In [ ]:
# Fig 2 – Hotel Star Rating Distribution
fig, ax = plt.subplots(figsize=(8, 5))
star_counts = df_meta['star_rating'].dropna().value_counts().sort_index()
ax.bar(star_counts.index.astype(str), star_counts.values, color='#EF9F27', edgecolor='none', width=0.6)
ax.set_title('Hotel Star Rating Distribution', fontweight='bold', fontsize=13)
ax.set_xlabel('Star Rating', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.spines[['top','right']].set_visible(False)
for i, (x, v) in enumerate(zip(star_counts.index.astype(str), star_counts.values)):
    ax.text(i, v+15, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_star.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_star.png')

In [ ]:
# Fig 3 – Hotel Price Distribution (AUD)
fig, ax = plt.subplots(figsize=(9, 5))
price_data = df_meta[df_meta['price_in_aud'] > 0]['price_in_aud']
ax.hist(price_data, bins=50, color='#1D9E75', edgecolor='none', alpha=0.85)
ax.axvline(price_data.mean(), color='#D85A30', linestyle='--', linewidth=1.8, label=f'Mean: AUD {price_data.mean():.0f}')
ax.axvline(price_data.median(), color='#534AB7', linestyle='--', linewidth=1.8, label=f'Median: AUD {price_data.median():.0f}')
ax.set_title('Hotel Price Distribution (AUD)', fontweight='bold', fontsize=13)
ax.set_xlabel('Price (AUD)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_price.png')

In [ ]:
# Fig 4 – Tourism Activity Category Distribution
fig, ax = plt.subplots(figsize=(8, 6))
cat_counts = df_act['categories_as_string'].dropna().str.split(',').str[0].str.strip().value_counts().head(6)
colors_act = ['#185FA5','#378ADD','#1D9E75','#5DCAA5','#EF9F27','#D85A30']
wedges, texts, autotexts = ax.pie(cat_counts.values, labels=None, autopct='%1.1f%%',
    colors=colors_act, startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts: at.set_fontsize(10)
ax.set_title('Tourism Activity Category Distribution', fontweight='bold', fontsize=13)
ax.legend([f'{k} ({v})' for k,v in cat_counts.items()], loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
plt.tight_layout()
plt.savefig('fig_activity.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_activity.png')

In [ ]:
# Fig 5 – Top 10 Hotels by Number of Transactions
fig, ax = plt.subplots(figsize=(9, 5))
top_hotels = df_tx.groupby('Product')['Number Transactions'].sum().sort_values(ascending=False).head(10)
ax.barh(top_hotels.index[::-1], top_hotels.values[::-1], color='#378ADD', edgecolor='none', height=0.6)
ax.set_title('Top 10 Hotels by Number of Transactions', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Transactions', fontsize=11)
ax.spines[['top','right']].set_visible(False)
for i, v in enumerate(top_hotels.values[::-1]):
    ax.text(v+0.1, i, str(int(v)), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_top_hotels.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_top_hotels.png')

In [ ]:
# Fig 6 – Provider Market Share
fig, ax = plt.subplots(figsize=(7, 6))
provider_share = df_tx['Property Id'].str.split('-').str[0].value_counts()
wedges, texts, autotexts = ax.pie(provider_share.values, labels=provider_share.index,
    autopct='%1.1f%%', colors=['#185FA5','#1D9E75','#EF9F27','#D85A30'],
    wedgeprops=dict(edgecolor='white', linewidth=2), startangle=90)
for at in autotexts: at.set_fontsize(10)
ax.set_title('Provider Market Share', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('fig_provider.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_provider.png')

In [ ]:
# Fig 7 – Guest Rating vs Hotel Price
fig, ax = plt.subplots(figsize=(8, 5))
df_plot = df_meta[(df_meta['price_in_aud']>0)&(df_meta['guest_rating_score']>=0)&(df_meta['guest_rating_score']<=100)].copy()
sc = ax.scatter(df_plot['guest_rating_score'], df_plot['price_in_aud'],
    c=df_plot['star_rating'].fillna(0), cmap='YlOrRd', alpha=0.4, s=15, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Star Rating')
ax.set_title('Guest Rating vs Hotel Price', fontweight='bold', fontsize=13)
ax.set_xlabel('Guest Rating (0-100)', fontsize=11)
ax.set_ylabel('Price (AUD)', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_rating_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_rating_vs_price.png')

In [ ]:
# Fig 8 – Popularity Score vs Hotel Price
fig, ax = plt.subplots(figsize=(8, 5))
df_pop = df_meta[df_meta['price_in_aud']>0].copy()
ax.scatter(df_pop['popularity_score'], df_pop['price_in_aud'],
    color='#378ADD', alpha=0.3, s=12, edgecolors='none')
ax.set_title('Popularity Score vs Hotel Price', fontweight='bold', fontsize=13)
ax.set_xlabel('Popularity Score', fontsize=11)
ax.set_ylabel('Price (AUD)', fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('fig_popularity_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_popularity_vs_price.png')

## 4. Data Preprocessing

In [ ]:
# ── Preprocessing metadata for scoring ──
df = df_meta.copy()

# Convert data types
df['guest_rating_score'] = pd.to_numeric(df['guest_rating_score'], errors='coerce')
df['popularity_score']   = pd.to_numeric(df['popularity_score'],   errors='coerce')
df['price_in_aud']       = pd.to_numeric(df['price_in_aud'],       errors='coerce')
df['star_rating']        = pd.to_numeric(df['star_rating'],        errors='coerce')

# Filter valid data:
# price > 0 (not free/unavailable)
# rating within valid range 0–100
# no missing values in key columns
df_valid = df[
    (df['price_in_aud'] > 0) &
    (df['guest_rating_score'] >= 0) &
    (df['guest_rating_score'] <= 100)
].dropna(subset=['guest_rating_score', 'popularity_score']).copy()

print(f'Raw data        : {len(df)} rows')
print(f'Valid data      : {len(df_valid)} rows')
print(f'Removed         : {len(df) - len(df_valid)} rows (outlier / missing)')
display(df_valid[['name','country','city','star_rating',
                   'price_in_aud','guest_rating_score','popularity_score']].head(5))

## 5. Composite Scoring Model

Formula:
$$CS = 0.3 \times norm(Popularity) + 0.3 \times norm(GuestRating) + 0.3 \times norm(ReviewCount) + 0.1 \times (1 - norm(Price))$$

- **norm(x)** = Min-Max normalization → value range [0, 1]
- Weight 0.3 for popularity, rating, and review count (quality & market trustworthiness)
- Weight 0.1 for price affordability (cheaper = higher score)

In [ ]:
# ── Min-Max normalization function ──
def minmax_normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - mn) / (mx - mn)

# ── Normalize columns ──
df_valid['norm_popularity'] = minmax_normalize(df_valid['popularity_score'])
df_valid['norm_rating']     = minmax_normalize(df_valid['guest_rating_score'])
df_valid['norm_price']      = minmax_normalize(df_valid['price_in_aud'])

# ── Calculate Composite Score ──
W_POPULARITY = 0.4
W_RATING     = 0.4
W_PRICE      = 0.2

df_valid['composite_score'] = (
    W_POPULARITY * df_valid['norm_popularity'] +
    W_RATING     * df_valid['norm_rating'] +
    W_PRICE      * (1 - df_valid['norm_price'])   # 1-norm because cheaper = better
)

print('✅ Composite Score berhasil dihitung!')
print(f'   Range skor: {df_valid["composite_score"].min():.4f} – {df_valid["composite_score"].max():.4f}')
print(f'   Rata-rata : {df_valid["composite_score"].mean():.4f}')

In [ ]:
# Top 10 Hotel Recommendations
cols_show = ['name','country','city','star_rating','price_in_aud',
             'guest_rating_score','composite_score']

top10 = df_valid.nlargest(10, 'composite_score')[cols_show].reset_index(drop=True)
top10.index += 1
top10.columns = ['Hotel Name','Country','City','Stars','Price (AUD)','Rating (0-100)','CS Score']
top10['CS Score'] = top10['CS Score'].round(4)

print('TOP 10 HOTELS BY COMPOSITE SCORE:')
display(top10)

In [ ]:
# ── Top 10 Visualization ──
fig, ax = plt.subplots(figsize=(12, 6))

names_short = [n[:35]+'...' if len(n)>35 else n for n in top10['Hotel Name']]
colors_bar  = ['#185FA5' if i == 0 else '#378ADD' if i < 3 else '#85B7EB'
               for i in range(len(top10))]

bars = ax.barh(names_short[::-1], top10['CS Score'].values[::-1],
               color=colors_bar[::-1], edgecolor='none')
ax.set_title('Top 10 Hotels – Composite Score (CS)', fontweight='bold', fontsize=13)
ax.set_xlabel('Composite Score')
ax.set_xlim(0.9, 1.01)
ax.spines[['top','right']].set_visible(False)

for bar, val in zip(bars, top10['CS Score'].values[::-1]):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('fig_top10_scoring.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_top10_scoring.png')

## 6. Recommendation Function by Destination

In [ ]:
def recommend_hotel(country=None, city=None, top_n=5):
    """
    Display hotel recommendations based on Composite Score.

    Parameters:
    - country : str, filter by country (optional)
    - city    : str, filter by city (optional)
    - top_n   : int, number of recommendations to display
    """
    df_filter = df_valid.copy()

    if country:
        df_filter = df_filter[df_filter['country'].str.lower().str.contains(country.lower())]
    if city:
        df_filter = df_filter[df_filter['city'].str.lower().str.contains(city.lower())]

    if df_filter.empty:
        print(f'❌ No data found for: country={country}, city={city}')
        return

    result = df_filter.nlargest(top_n, 'composite_score')[
        ['name','country','city','star_rating','price_in_aud',
         'guest_rating_score','composite_score']
    ].reset_index(drop=True)
    result.index += 1
    result.columns = ['Hotel Name','Country','City','Stars','Price (AUD)','Rating','CS Score']
    result['CS Score'] = result['CS Score'].round(4)

    title = f'Top {top_n} Hotels'
    if country: title += f' in {country}'
    if city:    title += f', {city}'
    print(f'{title}:')
    display(result)

# ── Example usage ──
recommend_hotel(country='Spain', top_n=5)

In [ ]:
# Try another country
recommend_hotel(country='Italy', top_n=5)

In [ ]:
# All available countries in valid data
print('Available countries in valid data:')
print(sorted(df_valid['country'].unique()))

## 7. Additional Analysis – CS Score Distribution by Star Segment

In [ ]:
# Complete preprocessing - run this first!
for col in ['guest_rating_score','popularity_score','price_in_aud','star_rating','guest_rating_count']:
    df_valid[col] = pd.to_numeric(df_valid[col], errors='coerce')

df_valid['guest_rating_count'] = df_valid['guest_rating_count'].fillna(0)

def minmax(s):
    mn, mx = s.min(), s.max()
    return (s-mn)/(mx-mn) if mx!=mn else pd.Series([0.5]*len(s), index=s.index)

df_valid['norm_pop']    = minmax(df_valid['popularity_score'])
df_valid['norm_rating'] = minmax(df_valid['guest_rating_score'])
df_valid['norm_price']  = minmax(df_valid['price_in_aud'])
df_valid['norm_review'] = minmax(df_valid['guest_rating_count'])

df_valid['score_cs'] = (
    0.3 * df_valid['norm_pop'] +
    0.3 * df_valid['norm_rating'] +
    0.3 * df_valid['norm_review'] +
    0.1 * (1 - df_valid['norm_price'])
)

print(f'✅ All columns ready!')
print(f'score_cs range: {df_valid["score_cs"].min():.4f} - {df_valid["score_cs"].max():.4f}')

In [ ]:
# Fig – Average CS Score by Star Rating
fig, ax = plt.subplots(figsize=(8, 5))
cs_by_star = df_valid.groupby('star_rating')['score_cs'].mean().sort_index()
ax.bar(cs_by_star.index.astype(str), cs_by_star.values, color='#378ADD', edgecolor='none', width=0.6)
ax.set_title('Average CS Score by Star Rating', fontweight='bold', fontsize=13)
ax.set_xlabel('Star Rating', fontsize=11)
ax.set_ylabel('Average CS Score', fontsize=11)
ax.set_ylim(0, 1)
ax.spines[['top','right']].set_visible(False)
for i, (x, v) in enumerate(zip(cs_by_star.index.astype(str), cs_by_star.values)):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_cs_by_star.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_cs_by_star.png')

In [ ]:
# Fig – Top 8 Countries by Average CS Score
fig, ax = plt.subplots(figsize=(9, 5))
cs_by_country = df_valid.groupby('country')['score_cs'].mean().sort_values(ascending=False).head(8)
ax.barh(cs_by_country.index[::-1], cs_by_country.values[::-1], color='#1D9E75', edgecolor='none', height=0.6)
ax.set_title('Top 8 Countries – Average CS Score', fontweight='bold', fontsize=13)
ax.set_xlabel('Average CS Score', fontsize=11)
ax.set_xlim(0, 1)
ax.spines[['top','right']].set_visible(False)
for i, v in enumerate(cs_by_country.values[::-1]):
    ax.text(v+0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_cs_by_country.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_cs_by_country.png')

## 8. Insights and Conclusion

In [ ]:
print('KEY INSIGHTS SUMMARY')

print(f"""
1. DATASET
   - Metadata    : {len(df_meta)} hotels from {df_meta['country'].nunique()} countries
   - Transaction : {len(df_tx)} transactions, {df_tx['Product'].nunique()} unique hotels
   - Activity    : {len(df_act)} tourism activities
   - Valid data for scoring: {len(df_valid)} properties

2. MARKET DISTRIBUTION
   - Most represented: {df_meta['country'].value_counts().index[0]} ({df_meta['country'].value_counts().iloc[0]} properties)
   - Dominant segment: 3-star hotels ({df_meta['star_rating'].value_counts().get(3.0, 0)} properties)
   - Average price: AUD {df_meta[df_meta['price_in_aud']>0]['price_in_aud'].mean():.2f}

3. TRANSACTIONS
   - Most booked hotel: {df_tx.groupby('Product')['Number Transactions'].sum().idxmax()}
     ({df_tx.groupby('Product')['Number Transactions'].sum().max()} transactions in 3 days)
   - Largest provider: {df_tx['Property Id'].str.split('-').str[0].value_counts().index[0]}

4. COMPOSITE SCORING
   - Best hotel: {df_valid.nlargest(1,'composite_score')['name'].values[0]}
   - Highest CS Score: {df_valid['composite_score'].max():.4f}
   - Key finding: High price does NOT guarantee high rating.
     Quality hotels can be found across various price ranges.
""")

---
## 9. Method Comparison: Composite Scoring vs TF-IDF CBF
> **Novelty:** Transaction-Validated Composite Scoring [Yilmaz et al., 2023]

Adding Content-Based Filtering (TF-IDF) as a baseline and evaluating both using **Precision@K, Recall@K, NDCG@K**.

In [ ]:
!pip install scikit-learn -q
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.patches as mpatches
print('✅ Ready!')

In [ ]:
# Ground truth: top 25% hotels by review count
threshold = df_valid['guest_rating_count'].quantile(0.75)
df_valid['is_relevant'] = (df_valid['guest_rating_count'] >= threshold).astype(int)
print(f'Ground truth threshold : {threshold:.0f} reviews')
print(f'Relevant hotels        : {df_valid["is_relevant"].sum():,} ({df_valid["is_relevant"].mean()*100:.1f}%)')

In [ ]:
# Update Composite Score (add review count component)
df_valid['norm_review'] = (df_valid['guest_rating_count'] - df_valid['guest_rating_count'].min()) / \
                          (df_valid['guest_rating_count'].max() - df_valid['guest_rating_count'].min())
df_valid['score_cs'] = (
    0.3 * df_valid['norm_pop'] +
    0.3 * df_valid['norm_rating'] +
    0.3 * df_valid['norm_review'] +
    0.1 * (1 - df_valid['norm_price'])
)
print('CS Formula: 0.3×Popularity + 0.3×Rating + 0.3×ReviewCount + 0.1×(1-Price)')
print(f'Score range: {df_valid["score_cs"].min():.4f} - {df_valid["score_cs"].max():.4f}')

top10_cs = df_valid.nlargest(10,'score_cs')[['name','country','star_rating','price_in_aud','guest_rating_score','guest_rating_count','score_cs','is_relevant']].reset_index(drop=True)
top10_cs.index += 1
top10_cs.columns = ['Hotel Name','Country','Stars','Price (AUD)','Rating','Reviews','CS Score','Relevant?']
top10_cs['Relevant?'] = top10_cs['Relevant?'].map({1:'✅ Yes', 0:'❌ No'})
display(top10_cs)

In [ ]:
# TF-IDF CBF Model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def build_features(row):
    parts = [str(row.get(c,'')) for c in ['category','country','city','star_rating','destination_keywords','tags']]
    return ' '.join([p for p in parts if p!='nan']).lower()

# Reset index dulu biar sinkron
df_valid = df_valid.reset_index(drop=True)

df_valid['features'] = df_valid.apply(build_features, axis=1)
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_valid['features'])

# Gunakan positional index, bukan label index
relevant_idx = df_valid[df_valid['is_relevant']==1].index.tolist()
user_profile = np.asarray(tfidf_matrix[relevant_idx].mean(axis=0))
df_valid['score_tfidf'] = cosine_similarity(user_profile, tfidf_matrix).flatten()

top10_tfidf = df_valid.nlargest(10,'score_tfidf')[['name','country','star_rating','price_in_aud','guest_rating_score','score_tfidf','is_relevant']].reset_index(drop=True)
top10_tfidf.index += 1
top10_tfidf.columns = ['Hotel Name','Country','Stars','Price (AUD)','Rating','TF-IDF Score','Relevant?']
top10_tfidf['Relevant?'] = top10_tfidf['Relevant?'].map({1:'✅ Yes', 0:'❌ No'})
print('Top 10 Hotels – TF-IDF CBF:')
display(top10_tfidf)

In [ ]:
# Evaluation Metrics
def precision_at_k(df, col, k): return df.nlargest(k, col)['is_relevant'].sum() / k
def recall_at_k(df, col, k):
    t = df['is_relevant'].sum()
    return 0 if t==0 else df.nlargest(k, col)['is_relevant'].sum() / t
def ndcg_at_k(df, col, k):
    top = df.nlargest(k, col)['is_relevant'].values
    dcg  = sum(r/np.log2(i+2) for i,r in enumerate(top))
    idcg = sum(1/np.log2(i+2) for i in range(min(int(df['is_relevant'].sum()), k)))
    return dcg/idcg if idcg>0 else 0

results = []
for method, col in [('Composite Scoring (Ours)','score_cs'),('TF-IDF CBF','score_tfidf')]:
    for k in [5,10,20]:
        results.append({'Method':method,'K':k,
            'Precision@K':round(precision_at_k(df_valid,col,k),4),
            'Recall@K':   round(recall_at_k(df_valid,col,k),4),
            'NDCG@K':     round(ndcg_at_k(df_valid,col,k),4)})

df_results = pd.DataFrame(results)
display(df_results)

df_k10 = df_results[df_results['K']==10][['Method','Precision@K','Recall@K','NDCG@K']].reset_index(drop=True)
df_k10.index += 1
print('\n=== COMPARISON @K=10 ===')
display(df_k10)

cs = df_k10[df_k10['Method']=='Composite Scoring (Ours)']['NDCG@K'].values[0]
tf = df_k10[df_k10['Method']=='TF-IDF CBF']['NDCG@K'].values[0]
print(f'\n✅ Composite Scoring: NDCG@10={cs} | TF-IDF: NDCG@10={tf} → {cs/tf:.1f}x better')

In [ ]:
# Fig 9 – Method Comparison Visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Hotel Recommendation Method Comparison\nComposite Scoring vs Content-Based Filtering (TF-IDF) @K=10',
             fontweight='bold', fontsize=11)
df_k10_plot = df_results[df_results['K']==10]
for ax, metric in zip(axes, ['Precision@K','Recall@K','NDCG@K']):
    vals = df_k10_plot[metric].values
    colors = ['#185FA5' if vals[0]>=vals[1] else '#85B7EB',
              '#EF9F27' if vals[1]>vals[0] else '#F5C97A']
    bars = ax.bar(['Composite\nScoring','CBF\n(TF-IDF)'], vals, color=colors, edgecolor='none', width=0.5)
    ax.set_title(metric, fontweight='bold', fontsize=11)
    ax.set_ylim(0, max(vals.max()*1.35, 0.1))
    ax.spines[['top','right']].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.4f}',
                ha='center', fontsize=10, fontweight='bold')
fig.legend(handles=[
    mpatches.Patch(color='#185FA5', label='Composite Scoring (Ours)'),
    mpatches.Patch(color='#EF9F27', label='Content-Based Filtering (TF-IDF)')
], loc='lower center', ncol=2, fontsize=10, frameon=False)
plt.tight_layout(rect=[0,0.1,1,1])
plt.savefig('fig_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_comparison.png')